In [14]:
%load_ext autoreload
%autoreload 2
from src.data_loading import load_data, PROCESSED_DATA_PATH
import pandas as pd
from sklearn.metrics import roc_auc_score
from src.evaluation import *
from src.training import run_lgbm_cv
import src.secondary_tables as st
application_train, application_test = load_data()
y = application_train[TARGET_COL]

auc_results = {}

def record_auc(stage, X, results):
    auc_results[stage] = {
        "num_features": X.shape[1],
        "mean_auc": results["mean_auc"],
        "std_auc": results["std_auc"],
        "oof_auc": roc_auc_score(y, results["oof_preds"]),
    }
    auc_table = pd.DataFrame.from_dict(auc_results, orient="index").reset_index()
    return auc_table.rename(columns={"index": "stage"})


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
from src.data_loading import ROOT, RAW_DATA_PATH
bureau = pd.read_csv(RAW_DATA_PATH / 'bureau.csv')
application_train_1, application_test_1 = st.add_bureau(application_train, application_test, bureau)
X_1 = application_train_1[[c for c in application_train_1.columns if c not in [TARGET_COL, ID_COL]]]
results = run_lgbm_cv(X_1, y)
summarize_cv_results(results, y)
record_auc("bureau", X_1, results)

/Users/ryanyao/Quant Projects/Home_Credit/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/ryanyao/Quant Projects/Home_Credit/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/ryanyao/Quant Projects/Home_Credit/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/ryanyao/Quant Projects/Home_Credit/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/ryanyao/Quant Projects/Home_Credit/.venv/lib/python3.14/site-packages/sklearn/utils/v

mean_auc: 0.7687409437865169
std_auc: 0.003662369574692071
oof_auc: 0.7687337721920476


,stage,num_features,mean_auc,std_auc,oof_auc
0,bureau,189,0.768741,0.003662,0.768734


In [16]:
bureau = pd.read_csv(RAW_DATA_PATH / 'bureau.csv')
bureau = st.add_bureau_balance(bureau)
application_train_2, application_test_2 = st.add_bureau(application_train, application_test, bureau)
X_2 = application_train_2[[c for c in application_train_2.columns if c not in [TARGET_COL, ID_COL]]]
results = run_lgbm_cv(X_2, y)
summarize_cv_results(results, y)
application_train_2.to_csv(PROCESSED_DATA_PATH / 'application_train_2.csv')
application_test_2.to_csv(PROCESSED_DATA_PATH / 'application_test_2.csv')
record_auc("bureau + bureau_balance", X_2, results)

/Users/ryanyao/Quant Projects/Home_Credit/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/ryanyao/Quant Projects/Home_Credit/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/ryanyao/Quant Projects/Home_Credit/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/ryanyao/Quant Projects/Home_Credit/.venv/lib/python3.14/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/Users/ryanyao/Quant Projects/Home_Credit/.venv/lib/python3.14/site-packages/sklearn/utils/v

mean_auc: 0.7679890834720554
std_auc: 0.0037343169448552137
oof_auc: 0.7679672152902898


,stage,num_features,mean_auc,std_auc,oof_auc
0,bureau,189,0.768741,0.003662,0.768734
1,bureau + bureau_balance,219,0.767989,0.003734,0.767967


In [17]:
application_train_3, application_test_3 = st.add_previous_application(application_train_2, application_test_2)
X_3 = application_train_3[[c for c in application_train_3.columns if c not in [TARGET_COL, ID_COL]]]
results = run_lgbm_cv(X_3, y)
summarize_cv_results(results, y)
record_auc("bureau + bureau_balance + previous_application", X_3, results)

mean_auc: 0.7758350695092464
std_auc: 0.004205257688702371
oof_auc: 0.7758065106688145


,stage,num_features,mean_auc,std_auc,oof_auc
0,bureau,189,0.768741,0.003662,0.768734
1,bureau + bureau_balance,219,0.767989,0.003734,0.767967
2,bureau + bureau_balance + previous_application,331,0.775835,0.004205,0.775807


In [18]:
application_train_4, application_test_4 = st.add_pos_cash_balance(application_train_3, application_test_3)
X_4 = application_train_4[[c for c in application_train_4.columns if c not in [TARGET_COL, ID_COL]]]
results = run_lgbm_cv(X_4, y)
summarize_cv_results(results, y)
record_auc("bureau + bureau_balance + previous_application + POS_CASH_balance", X_4, results)

mean_auc: 0.7781311369589207
std_auc: 0.0038110986004461184
oof_auc: 0.7781213889641689


,stage,num_features,mean_auc,std_auc,oof_auc
0,bureau,189,0.768741,0.003662,0.768734
1,bureau + bureau_balance,219,0.767989,0.003734,0.767967
2,bureau + bureau_balance + previous_application,331,0.775835,0.004205,0.775807
3,bureau + bureau_balance + previous_application...,350,0.778131,0.003811,0.778121


In [19]:
application_train_5, application_test_5 = st.add_credit_card_balance(application_train_4, application_test_4)
X_5 = application_train_5[[c for c in application_train_5.columns if c not in [TARGET_COL, ID_COL]]]
results = run_lgbm_cv(X_5, y)
summarize_cv_results(results, y)
record_auc("bureau + bureau_balance + previous_application + POS_CASH_balance + credit_card_balance", X_5, results)

mean_auc: 0.7789405601898505
std_auc: 0.003735426975563719
oof_auc: 0.7789308006273499


,stage,num_features,mean_auc,std_auc,oof_auc
0,bureau,189,0.768741,0.003662,0.768734
1,bureau + bureau_balance,219,0.767989,0.003734,0.767967
2,bureau + bureau_balance + previous_application,331,0.775835,0.004205,0.775807
3,bureau + bureau_balance + previous_application...,350,0.778131,0.003811,0.778121
4,bureau + bureau_balance + previous_application...,444,0.778941,0.003735,0.778931


In [20]:
application_train_6, application_test_6 = st.add_installments_payments(application_train_5, application_test_5)
application_train_6.to_csv(PROCESSED_DATA_PATH / 'application_train_with_secondary_tables.csv')
application_test_6.to_csv(PROCESSED_DATA_PATH / 'application_test_with_secondary_tables.csv')
X_6 = application_train_6[[c for c in application_train_6.columns if c not in [TARGET_COL, ID_COL]]]
results = run_lgbm_cv(X_6, y)
summarize_cv_results(results, y)
record_auc("all secondary tables", X_6, results)

mean_auc: 0.7826111073113771
std_auc: 0.0032210102478750723
oof_auc: 0.782603192099121


,stage,num_features,mean_auc,std_auc,oof_auc
0,bureau,189,0.768741,0.003662,0.768734
1,bureau + bureau_balance,219,0.767989,0.003734,0.767967
2,bureau + bureau_balance + previous_application,331,0.775835,0.004205,0.775807
3,bureau + bureau_balance + previous_application...,350,0.778131,0.003811,0.778121
4,bureau + bureau_balance + previous_application...,444,0.778941,0.003735,0.778931
5,all secondary tables,484,0.782611,0.003221,0.782603
